<a href="https://colab.research.google.com/github/Mohamed-Yaseer-S/Nan-Mudhalvan/blob/main/SQL_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import sqlite3
import pandas as pd
import os
import gradio as gr
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet

# 1. DATABASE SETUP
def setup_database():
    conn = sqlite3.connect('nan_mudhalvan.db', check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS employees")
    cursor.execute("CREATE TABLE employees (id INT, name TEXT, dept TEXT, salary INT)")
    cursor.execute("INSERT INTO employees VALUES (1,'Ravi','IT',75000),(2,'Priya','HR',65000),(3,'Kumar','IT',80000)")
    conn.commit()
    return conn

db_conn = setup_database()

# 2. MODEL LOADING
model_id = "ibm-granite/granite-3.3-2b-instruct"
print("🔄 Loading IBM Granite 3.3 2B...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=100, temperature=0.1)

# 3. PDF GENERATION
def create_pdf_report(question, sql, df):
    filename = "Granite_SQL_Report.pdf"
    doc = SimpleDocTemplate(filename, pagesize=letter)
    styles = getSampleStyleSheet()
    content = [Paragraph("Nan Mudhalvan: AI SQL Report", styles['Title']),
               Paragraph(f"<b>Question:</b> {question}", styles['Normal']),
               Paragraph(f"<b>SQL:</b> {sql}", styles['Code'])]
    table_data = [df.columns.to_list()] + df.values.tolist()
    t = Table(table_data)
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),'#4F81BD'),('GRID',(0,0),(-1,-1),1,'#000000')]))
    content.append(t)
    doc.build(content)
    return os.path.abspath(filename)

# 4. MAIN PROCESS FUNCTION (With Social Logic)
def run_ai_logic(question):
    try:
        prompt = f"<|im_start|>user\nTable: employees(id, name, dept, salary)\nGenerate ONLY the SQLite SQL for: {question}\n<|im_end|>\n<|im_start|>assistant\nSELECT"
        raw_output = pipe(prompt)[0]['generated_text']
        sql_part = raw_output.split("assistant\n")[-1].split("<|im_end|>")[0].strip()
        sql_clean = (sql_part if sql_part.upper().startswith("SELECT") else "SELECT " + sql_part).split(';')[0] + ";"

        df_results = pd.read_sql_query(sql_clean, db_conn)
        pdf_path = create_pdf_report(question, sql_clean, df_results)

        # --- SOCIAL SHARE BUTTONS LOGIC ---
        share_msg = f"I generated this SQL using IBM Granite AI: {sql_clean}"
        wa_link = f"https://api.whatsapp.com/send?text={share_msg}"
        fb_link = f"https://www.facebook.com/sharer/sharer.php?u=https://huggingface.co/ibm-granite&quote={share_msg}"

        # Creating HTML buttons for the UI
        social_html = f"""
        <div style="display: flex; gap: 10px; margin-top: 10px;">
            <a href="{wa_link}" target="_blank" style="background-color: #25D366; color: white; padding: 10px 20px; text-decoration: none; border-radius: 5px; font-weight: bold;">WhatsApp Share</a>
            <a href="{fb_link}" target="_blank" style="background-color: #1877F2; color: white; padding: 10px 20px; text-decoration: none; border-radius: 5px; font-weight: bold;">Facebook Share</a>
        </div>
        """

        return sql_clean, df_results, pdf_path, social_html

    except Exception as e:
        return f"Error: {str(e)}", pd.DataFrame(), None, ""

# 5. GRADIO UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 IBM Granite AI: English to SQL")

    with gr.Row():
        with gr.Column(scale=3):
            input_box = gr.Textbox(label="Enter Question", placeholder="e.g. List all employees")
            submit_btn = gr.Button("Generate & Execute", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### Share Output")
            share_component = gr.HTML("Submit a query to see share options")

    with gr.Tabs():
        with gr.TabItem("📊 Results"):
            sql_display = gr.Code(label="Generated SQL", language="sql")
            table_display = gr.DataFrame()
        with gr.TabItem("📥 Export"):
            pdf_display = gr.File(label="Download PDF")

    submit_btn.click(run_ai_logic, inputs=input_box, outputs=[sql_display, table_display, pdf_display, share_component])

demo.launch(share=True)

🔄 Loading IBM Granite 3.3 2B...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_7085/3161739841.py:75: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2ce2a0ab8f1bfa277a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
